# 🌿 Ecosystem Simulator — Notebook Runner
Run the simulation from a `params.json` exported from the Streamlit frontend, then save a `.pkl` that can be loaded back into the app.

This rewritten version infers the number of elements from `params.json` and supports arbitrary `N_ELEM` (including 11 elements).#%% md
# 🌿 Ecosystem Simulator — Notebook Runner
Run the simulation from a `params.json` exported from the Streamlit frontend, then save a `.pkl` that can be loaded back into the app.

This rewritten version infers the number of elements from `params.json` and supports arbitrary `N_ELEM` (including 11 elements).

In [2]:
import sys, os
PROJECT_ROOT = os.path.abspath("..")
sys.path.insert(0, PROJECT_ROOT)

import json
import pickle
import numpy as np
import tensorflow as tf
from datetime import datetime
from tqdm.notebook import tqdm

try:
    from hybridmodel import HybridEcosystem
except ImportError:
    from models.hybridmodel import HybridEcosystem

try:
    from google.colab import files
    IN_COLAB = True
except Exception:
    IN_COLAB = False

print(f"Project root: {PROJECT_ROOT}")
print(f"Running in Colab: {IN_COLAB}")

C:\Users\f.andrade\AppData\Local\anaconda3\envs\elementalworld\lib\site-packages\requests\__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


ModuleNotFoundError: No module named 'tqdm'

## 1 — Load parameters

In [ ]:
PARAMS_PATH = "params.json"

with open(PARAMS_PATH) as f:
    params = json.load(f)

N_SPP = int(params["N_SPP"])
H = int(params["H"])
W = int(params["W"])
MAX_AGENTS = int(params["MAX_AGENTS"])
N_STEPS = int(params["N_STEPS"])
SEED = int(params["SEED"])
NSEEDS = int(params.get("NSEEDS", 1))
scalar_interval = int(params.get("scalar_interval", 20))
snapshot_interval = int(params.get("snapshot_interval", 50))

growth_rate = float(params["growth_rate"])
respiration_rate = float(params["respiration_rate"])
turnover_rate = float(params["turnover_rate"])
mineralization_rate = float(params["mineralization_rate"])
seed_cost = float(params["seed_cost"])
seed_mass = float(params["seed_mass"])
K_biomass = float(params["K_biomass"])
soil_input_rate = float(params["soil_input_rate"])
sigma_threshold = float(params["sigma_threshold"])
soil_pool_mean = float(params["soil_pool_mean"])
soil_pool_std = float(params["soil_pool_std"])
soil_ratio_noise = float(params["soil_ratio_noise"])
input_drift_scale = float(params["input_drift_scale"])
seed_range_scale = float(params.get("seed_range_scale", 10.0))
seed_range_alpha = float(params.get("seed_range_alpha", 1.0))

catastrophe_interval = int(params.get("catastrophe_interval", 200))
catastrophe_mortality = float(params.get("catastrophe_mortality", 0.4))
weak_disturbance_interval = int(params.get("weak_disturbance_interval", 0))
weak_disturbance_mortality = float(params.get("weak_disturbance_mortality", 0.4))
strong_disturbance_interval = int(params.get("strong_disturbance_interval", 0))
strong_disturbance_mortality = float(params.get("strong_disturbance_mortality", 0.4))
p_disturbance = float(params.get("p_disturbance", 0.01))
disturbance_strength = float(params.get("disturbance_strength", 0.7))
demo_noise_std = float(params.get("demo_noise_std", 0.003))

env_field_persistence = float(params.get("env_field_persistence", 0.85))
env_field_smoothing_passes = int(params.get("env_field_smoothing_passes", 2))
shock_field_persistence = float(params.get("shock_field_persistence", 0.85))
shock_field_smoothing_passes = int(params.get("shock_field_smoothing_passes", 2))

temperature_mean = float(params.get("temperature_mean", 0.5))
temperature_amplitude = float(params.get("temperature_amplitude", 0.3))
temperature_period = int(params.get("temperature_period", 100))
temperature_phase = float(params.get("temperature_phase", 0.0))
temperature_spatial_strength = float(params.get("temperature_spatial_strength", 0.0))
temperature_growth_strength = float(params.get("temperature_growth_strength", 0.5))
temperature_respiration_strength = float(params.get("temperature_respiration_strength", 0.3))
temperature_mineralization_strength = float(params.get("temperature_mineralization_strength", 0.4))

spp_centers = np.array(params["spp_centers"], dtype=np.float32)
N_ELEM = int(spp_centers.shape[1])
sbr = np.array(params["soil_base_ratio"], dtype=np.float32)
sar = np.array(params["soil_availability_rate"], dtype=np.float32)
initial_seeds = [int(x) for x in params["initial_seeds"]]
seed_mass_by_species = np.array(
    params.get("seed_mass_by_species", [seed_mass] * N_SPP),
    dtype=np.float32
)

assert spp_centers.shape == (N_SPP, N_ELEM), \
    f"spp_centers must have shape ({N_SPP}, {N_ELEM}), got {spp_centers.shape}"
assert sbr.shape == (N_ELEM,), \
    f"soil_base_ratio must have shape ({N_ELEM},), got {sbr.shape}"
assert sar.shape == (N_ELEM,), \
    f"soil_availability_rate must have shape ({N_ELEM},), got {sar.shape}"
assert len(initial_seeds) == N_SPP, \
    f"initial_seeds must have length {N_SPP}, got {len(initial_seeds)}"
assert seed_mass_by_species.shape == (N_SPP,), \
    f"seed_mass_by_species must have shape ({N_SPP},), got {seed_mass_by_species.shape}"

if "spp_covariances" in params:
    SPP_COVARIANCES = np.array(params["spp_covariances"], dtype=np.float32)
else:
    cov_code = params.get(
        "cov_code",
        "np.array([np.eye(N_ELEM, dtype=np.float32) * 0.03 for _ in range(N_SPP)], dtype=np.float32)"
    )
    SPP_COVARIANCES = np.array(
        eval(cov_code, {"np": np, "N_SPP": N_SPP, "N_ELEM": N_ELEM}),
        dtype=np.float32
    )

assert SPP_COVARIANCES.shape == (N_SPP, N_ELEM, N_ELEM), \
    f"Expected covariance shape ({N_SPP}, {N_ELEM}, {N_ELEM}), got {SPP_COVARIANCES.shape}"

print(f"Loaded: {N_SPP} species | {N_ELEM} elements | {H}x{W} grid | {N_STEPS} steps | {NSEEDS} seed(s)")
print(f"spp_centers shape: {spp_centers.shape}")
print(f"soil_base_ratio shape: {sbr.shape}")
print(f"soil_availability_rate shape: {sar.shape}")
print(f"Covariance shape: {SPP_COVARIANCES.shape}")

## 2 — Model factory

In [ ]:
def _make_model():
    return HybridEcosystem(
        height=H,
        width=W,
        max_agents=MAX_AGENTS,
        niche_centers=spp_centers,
        niche_covariances=SPP_COVARIANCES,
        growth_rate=growth_rate,
        respiration_rate=respiration_rate,
        turnover_rate=turnover_rate,
        mineralization_rate=mineralization_rate,
        seed_cost=seed_cost,
        seed_mass=seed_mass,
        seed_mass_by_species=seed_mass_by_species,
        seed_range_scale=seed_range_scale,
        seed_range_alpha=seed_range_alpha,
        K_biomass=K_biomass,
        soil_base_ratio=sbr,
        soil_pool_mean=soil_pool_mean,
        soil_pool_std=soil_pool_std,
        soil_ratio_noise=soil_ratio_noise,
        soil_input_rate=soil_input_rate,
        soil_availability_rate=sar,
        input_drift_scale=input_drift_scale,
        sigma_threshold=sigma_threshold,
        catastrophe_interval=catastrophe_interval,
        catastrophe_mortality=catastrophe_mortality,
        p_disturbance=p_disturbance,
        disturbance_strength=disturbance_strength,
        demo_noise_std=demo_noise_std,
        weak_disturbance_interval=weak_disturbance_interval,
        weak_disturbance_mortality=weak_disturbance_mortality,
        strong_disturbance_interval=strong_disturbance_interval,
        strong_disturbance_mortality=strong_disturbance_mortality,
        env_field_persistence=env_field_persistence,
        env_field_smoothing_passes=env_field_smoothing_passes,
        shock_field_persistence=shock_field_persistence,
        shock_field_smoothing_passes=shock_field_smoothing_passes,
        temperature_mean=temperature_mean,
        temperature_amplitude=temperature_amplitude,
        temperature_period=temperature_period,
        temperature_phase=temperature_phase,
        temperature_spatial_strength=temperature_spatial_strength,
        temperature_growth_strength=temperature_growth_strength,
        temperature_respiration_strength=temperature_respiration_strength,
        temperature_mineralization_strength=temperature_mineralization_strength,
    )

## 3 — Run simulation

In [ ]:
def _run_one_seed(seed):
    tf.random.set_seed(seed)
    np.random.seed(seed)

    model = _make_model()
    soil_snap = model.soil.numpy().copy()

    for s_id, n in enumerate(initial_seeds):
        model.add_initial_seeds(count=n, species_id=s_id)

    history_biomass, history_agents, history_elements = [], [], []
    history_biomass_grid = []
    history_spp_biomass = [[] for _ in range(N_SPP)]
    history_spp_fitness = [[] for _ in range(N_SPP)]
    history_spp_dead_fitness_mean = [[] for _ in range(N_SPP)]
    history_spp_grid = [[] for _ in range(N_SPP)]
    history_deficit = [[] for _ in range(N_SPP)]
    history_spp_age = [[] for _ in range(N_SPP)]
    history_spp_elemental_dissimilarity = []

    for t in tqdm(range(N_STEPS), desc=f"Seed {seed}"):
        n_agents = model.step("mahalanobis")
        grid_total = model.get_biomass_grid()

        if t % scalar_interval == 0:
            history_biomass.append(float(np.mean(grid_total)))
            history_agents.append(int(n_agents.numpy()))
            history_elements.append(model.get_element_pools())
            deficit = model.get_nutrient_deficit()
            history_spp_elemental_dissimilarity.append(
                model.get_species_elemental_dissimilarity_index_tf()
            )

            for s_id in range(N_SPP):
                history_spp_biomass[s_id].append(
                    float(np.mean(model.get_species_biomass(s_id)))
                )
                fit = model.get_species_mean_fitness(s_id)
                history_spp_fitness[s_id].append(
                    float(fit) if fit is not None else None
                )
                dead_fit = model.get_species_mean_dead_fitness(s_id)
                history_spp_dead_fitness_mean[s_id].append(
                    float(dead_fit) if dead_fit is not None else None
                )
                history_deficit[s_id].append(np.array(deficit[s_id]).tolist())
                history_spp_age[s_id].append(model.get_species_mean_age(s_id))

            model.death_fitness_log.clear()

        if t % snapshot_interval == 0:
            history_biomass_grid.append(grid_total)
            for s_id in range(N_SPP):
                history_spp_grid[s_id].append(model.get_species_biomass(s_id))

    return {
        "soil_snap": soil_snap,
        "history_biomass": history_biomass,
        "history_agents": history_agents,
        "history_elements": history_elements,
        "history_biomass_grid": history_biomass_grid,
        "history_spp_biomass": history_spp_biomass,
        "history_spp_fitness": history_spp_fitness,
        "history_spp_dead_fitness_mean": history_spp_dead_fitness_mean,
        "history_spp_grid": history_spp_grid,
        "history_deficit": history_deficit,
        "history_spp_age": history_spp_age,
        "history_spp_elemental_dissimilarity": history_spp_elemental_dissimilarity,
        "final_state": {
            "agents": model.agents.numpy().copy(),
            "soil": model.soil.numpy().copy(),
            "step_count": N_STEPS,
        },
    }

In [ ]:
all_runs = [_run_one_seed(SEED + i) for i in range(NSEEDS)]
print(f"Finished {len(all_runs)} run(s).")

## 4 — Average ensemble & build payload

In [ ]:
def _avg_nullable(runs_list):
    arr = np.array(
        [[v if v is not None else np.nan for v in run] for run in runs_list],
        dtype=np.float64
    )
    mean = np.nanmean(arr, axis=0)
    return [None if np.isnan(v) else float(v) for v in mean]

def _avg_grids(runs_list):
    n_snaps = len(runs_list[0])
    return [np.mean([run[i] for run in runs_list], axis=0) for i in range(n_snaps)]

payload = {
    "parameters": params,
    "completed_steps": N_STEPS,
    "soil_snapshot": all_runs[0]["soil_snap"],

    "history_biomass": list(np.mean([r["history_biomass"] for r in all_runs], axis=0)),
    "history_spp_count": list(np.mean([r["history_agents"] for r in all_runs], axis=0)),
    "history_elements": list(np.mean([r["history_elements"] for r in all_runs], axis=0)),
    "history_biomass_grid": _avg_grids([r["history_biomass_grid"] for r in all_runs]),

    "history_spp_biomass": [
        _avg_nullable([r["history_spp_biomass"][s] for r in all_runs])
        for s in range(N_SPP)
    ],
    "history_spp_biomass_std": [
        list(np.std([r["history_spp_biomass"][s] for r in all_runs], axis=0))
        for s in range(N_SPP)
    ],
    "history_spp_fitness": [
        _avg_nullable([r["history_spp_fitness"][s] for r in all_runs])
        for s in range(N_SPP)
    ],
    "history_spp_dead_fitness_mean": [
        _avg_nullable([r["history_spp_dead_fitness_mean"][s] for r in all_runs])
        for s in range(N_SPP)
    ],
    "history_spp_biomass_grid": [
        _avg_grids([r["history_spp_grid"][s] for r in all_runs])
        for s in range(N_SPP)
    ],
    "history_deficit": [
        [
            list(np.mean([r["history_deficit"][s][t] for r in all_runs], axis=0))
            for t in range(len(all_runs[0]["history_deficit"][s]))
        ]
        for s in range(N_SPP)
    ],
    "history_spp_age": [
        _avg_nullable([r["history_spp_age"][s] for r in all_runs])
        for s in range(N_SPP)
    ],
    "history_spp_elemental_dissimilarity": list(
        np.mean([r["history_spp_elemental_dissimilarity"] for r in all_runs], axis=0)
    ),

    "final_states": [r["final_state"] for r in all_runs],
}

print("Payload ready.")
print(f"Biomass snapshots: {len(payload['history_biomass_grid'])}")
print(f"Scalar steps: {len(payload['history_biomass'])}")

## 5 — Save .pkl

In [3]:
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
outfile = f"sim_{timestamp}_standalone.pkl"

with open(outfile, "wb") as f:
    pickle.dump(payload, f)

print(f"✅ Saved {outfile}")
if IN_COLAB:
    files.download(outfile)

NameError: name 'payload' is not defined